# Multi-image FLAME head fit with Pixel3DMM

Personal non-commercial research workflow, owner decision 2026-10-02.
Read [Pixel3DMM](https://github.com/SimonGiebenhain/pixel3dmm),
[MICA](https://github.com/Zielon/MICA) and your [FLAME](https://flame.is.tue.mpg.de/)
licence before using their assets. Pixel3DMM is CC BY-NC 4.0; MICA,
insightface models and NVIDIA's renderer have separate restricted terms.
No models or upstream source are embedded here or redistributed by this notebook.

Register at FLAME, download **FLAME2020.zip** yourself and put it in your
Google Drive. Choose **Runtime > Change runtime type > GPU** (T4 supported; L4/A100 have more headroom).
Edit the parameter cell, then **Runtime > Run all**.
A separate Python 3.9 / CUDA 11.8 conda environment avoids restarting Colab.
Installation/native compilation and multi-GB weight downloads can take tens of minutes.
**The revised fitting/retry path has not been tested on Colab.**

Drive is mounted with read-only **usage**, not a read-only permission grant:
Colab's Drive mount can write. This notebook only reads the selected FLAME
archive(s), copies them into the VM and immediately unmounts Drive.
Never place/upload your photos in Drive for this workflow. When prompted,
select de-glassed `front.png`, `left.png`, `right.png`, `back.png` together
(.jpg/.jpeg also accepted). Left/right mean the subject's left/right profile.
Back is optional and omitted from fitting if no face is detected.
Front must preprocess successfully; undetectable profiles/back are skipped with warnings.

Photos run on Google's VM only, through user-operated uploads. No remote
inference service or logging service receives them. No previews are displayed.
Neutral head OBJ/PLY, JSON/NPZ parameters, cameras, fitted-view meshes and PNG
overlays download as **head_fit.zip**. Keep it locally in
**user-data/twin/head/flame/** (gitignored); never ship it with the web app.
This is a head intermediate, not the rigged `twin.glb` bundle.

Installation, upload, fitting and download share a `try/finally` cleanup.
Software, weights and FLAME stay in /content/dt-pixel3dmm-cache for retries.
The final privacy cell can explicitly delete that cache once you finish.
Save the browser download, then **Runtime > Disconnect and delete runtime**.
VM deletion is not secure erasure or a promise about Google's retention.
Clear notebook outputs before sharing. On hard kernel/VM termination,
cleanup cannot run; delete the runtime yourself.


In [ ]:
# Parameters: edit before Runtime > Run all.
FLAME_ZIP_DRIVE_PATH = "/content/drive/MyDrive/flame/FLAME2020.zip"
FLAME_VERSION = "2020"  # "2020" (default) or "2023" (no-jaw variant)
# For 2023, point FLAME_ZIP_DRIVE_PATH to FLAME2023.zip AND provide 2020:
# preprocessing/MICA and tracking landmark/mask assets still require 2020.
FLAME2020_ZIP_DRIVE_PATH = "/content/drive/MyDrive/flame/FLAME2020.zip"
ITERS = 1500
GLOBAL_ITERS = 1500
MAX_FIT_BATCH_SIZE = 1  # Small joint batches reduce peak memory on T4.


## GPU check

Fails before Drive access or image uploads unless a T4/L4/A100/H100 GPU is attached.
These GPU choices are notebook policy; no VRAM/performance guarantee is made.
FLAME 2023 fitting uses `use_flame2023=True` and `ignore_mica=True`.
The MICA stage still runs with its own FLAME 2020 model.


In [ ]:
import subprocess

if FLAME_VERSION not in {"2020", "2023"}:
    raise ValueError("FLAME_VERSION must be 2020 or 2023")
if any(not isinstance(value, int) or value < 1 for value in (ITERS, GLOBAL_ITERS, MAX_FIT_BATCH_SIZE)):
    raise ValueError("Iteration counts must be positive integers")
try:
    gpu_summary = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
    ).splitlines()[0]
except (FileNotFoundError, subprocess.CalledProcessError, IndexError) as error:
    raise RuntimeError("Select a T4, L4, A100 or H100 Colab GPU runtime") from error
gpu_name, gpu_memory = gpu_summary.rsplit(",", 1)
print(f"GPU: {gpu_name.strip()}, {int(gpu_memory)} MiB")
# Compile native extensions for the attached GPU, including T4 sm_75.
GPU_ARCHES = {"T4": "7.5", "L4": "8.9", "A100": "8.0", "H100": "9.0"}
GPU_ARCH = next((arch for name, arch in GPU_ARCHES.items() if name in gpu_name), None)
if GPU_ARCH is None or int(gpu_memory) < 15000:
    raise RuntimeError("A T4, L4, A100 or H100 GPU runtime is required")
if GPU_ARCH == "7.5":
    print("T4 detected: eager float32 execution, no bf16; use MAX_FIT_BATCH_SIZE=1 for memory headroom.")


In [ ]:
EMBEDDED_FILES = {'io_utils.py': '"""Original archive/privacy helpers; no third-party model code or data embedded."""\n\nfrom pathlib import Path\nimport gc\nimport shutil\nimport stat\nimport zipfile\n\n\ndef safe_extract(archive: Path, destination: Path) -> None:\n    """Reject traversal and symlinks before extracting a user-supplied model zip."""\n    destination = destination.resolve()\n    with zipfile.ZipFile(archive) as bundle:\n        for item in bundle.infolist():\n            target = (destination / item.filename).resolve()\n            if not target.is_relative_to(destination) or "\\\\" in item.filename:\n                raise ValueError("Unsafe zip member path")\n            if stat.S_ISLNK(item.external_attr >> 16):\n                raise ValueError("Zip symlinks are not allowed")\n        bundle.extractall(destination)\n\n\ndef stage_flame(archive: Path, assets: Path, version: str) -> None:\n    if version not in {"2020", "2023"}:\n        raise ValueError("FLAME_VERSION must be 2020 or 2023")\n    unpacked = assets / ("unpacked-" + version)\n    safe_extract(archive, unpacked)\n    filename = "generic_model.pkl" if version == "2020" else "flame2023_no_jaw.pkl"\n    candidates = list(unpacked.rglob(filename))\n    if len(candidates) != 1:\n        raise ValueError(f"FLAME {version} zip must contain exactly one {filename}")\n    target = assets / ("FLAME" + version)\n    target.mkdir(parents=True, exist_ok=True)\n    # Copy the model\'s sibling files, preserving upstream-supplied landmark assets.\n    shutil.copytree(candidates[0].parent, target, dirs_exist_ok=True)\n    shutil.rmtree(unpacked)\n    if version == "2020":\n        required = ["generic_model.pkl", "landmark_embedding.npy", "FLAME_masks/FLAME_masks.pkl"]\n        missing = [name for name in required if not (target / name).is_file()]\n        if missing:\n            raise ValueError("FLAME 2020 model/landmark/mask assets missing: " + ", ".join(missing))\n\n\ndef upload_views(names) -> dict:\n    result = {}\n    for name in names:\n        path = Path(name)\n        view = path.stem.lower()\n        if view not in {"front", "left", "right", "back"} or path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:\n            raise ValueError("Use front/left/right/back with .png, .jpg or .jpeg extensions")\n        if view in result:\n            raise ValueError("Only one image per view is allowed")\n        result[view] = path.name\n    if "front" not in result:\n        raise ValueError("Front is required; left/right/back are optional")\n    return result\n\n\ndef cleanup_session(path: Path, parent: Path = Path("/content")) -> None:\n    resolved = path.resolve()\n    if resolved.parent != parent.resolve() or not resolved.name.startswith("dt-pixel3dmm-session-"):\n        raise ValueError("Refusing cleanup outside the private Pixel3DMM session")\n    if resolved.exists():\n        shutil.rmtree(resolved)\n    gc.collect()\n\n\ndef download_and_wait(path: Path) -> None:\n    """Same completion convention as the TRELLIS notebook: await browser memory."""\n    import IPython.display\n    from google.colab import files, output\n    from unittest.mock import patch\n\n    original = IPython.display.display\n    instrumented = False\n\n    def display_with_completion(*objects, **kwargs):\n        nonlocal instrumented\n        changed = []\n        for item in objects:\n            if isinstance(item, IPython.display.Javascript) and item.data.strip().startswith("download("):\n                item = IPython.display.Javascript("window.dtHeadDownload = " + item.data)\n                instrumented = True\n            changed.append(item)\n        return original(*changed, **kwargs)\n\n    with patch.object(IPython.display, "display", display_with_completion):\n        files.download(str(path))\n    if not instrumented:\n        raise RuntimeError("Colab download API changed; completion wrapper needs updating")\n    output.eval_js("window.dtHeadDownload.then(() => true)", timeout_sec=600)\n', 'camera.py': '"""Numeric camera export for Pixel3DMM\'s OpenGL pinhole model (synthetic-testable)."""\n\nimport numpy as np\n\n\ndef camera_matrices(checkpoint: dict) -> tuple:\n    size = np.asarray(checkpoint["img_size"], dtype=int).reshape(2)\n    if size[0] != size[1]:\n        raise ValueError("This adapter requires square cropped images")\n    side = int(size[0])\n    camera = checkpoint["camera"]\n    focal = float(np.asarray(camera["fl"]).reshape(-1)[0]) * side\n    pp = np.asarray(camera["pp"]).reshape(-1, 2)[0]\n    center = (1 + pp) * (side / 2 + 0.5)\n    intrinsics = np.array([[focal, 0, center[0]], [0, focal, center[1]], [0, 0, 1]])\n    base = np.eye(4)\n    base[:3, :3] = np.asarray(camera["R_base_0"]).reshape(-1, 3, 3)[0]\n    base[:3, 3] = np.asarray(camera["t_base_0"]).reshape(-1, 3)[0]\n    head = np.eye(4)\n    head[:3, :3] = np.asarray(checkpoint["flame"]["R_rotation_matrix"]).reshape(-1, 3, 3)[0]\n    head[:3, 3] = np.asarray(checkpoint["flame"]["t"]).reshape(-1, 3)[0]\n    extrinsics = base @ head\n    if not np.isfinite(intrinsics).all() or not np.isfinite(extrinsics).all() or focal <= 0:\n        raise ValueError("Invalid fitted camera")\n    return intrinsics, extrinsics, side\n\n\ndef projection_matrix(intrinsics: np.ndarray, side: int) -> np.ndarray:\n    """Near/far = 0.1/5 m, matching upstream tracking\'s rasterizer.\n\n    Pixel-array row 0 corresponds to NDC y=-1: negative fy is deliberate.\n    This matrix uses pixel edge coordinates; pixel i\'s center is i+0.5.\n    """\n    near, far = 0.1, 5.0\n    result = np.zeros((4, 4), dtype=np.float32)\n    result[0, 0] = 2 * intrinsics[0, 0] / side\n    result[1, 1] = -2 * intrinsics[1, 1] / side\n    result[0, 2] = 1 - 2 * intrinsics[0, 2] / side\n    result[1, 2] = 1 - 2 * intrinsics[1, 2] / side\n    result[2, 2] = -(far + near) / (far - near)\n    result[2, 3] = -2 * far * near / (far - near)\n    result[3, 2] = -1\n    return result\n', 'setup_runtime.py': '"""Colab-only installation adapter. Downloads licensed sources/assets into the VM.\n\nNo upstream code, model, credentials or personal data belongs in this repository.\nSources checked 2026-10-02; full CUDA/Colab execution remains untested.\n"""\n\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport urllib.request\n\nfrom io_utils import safe_extract, stage_flame\n\nPIXEL_REVISION = "fcd1fa973c7715b02a8948dfc679dff53cf85924"\nMICA_REVISION = "af22e7a5810d474bc28a1433db533723d6bd2b07"\nFACER_REVISION = "ddd35c76ff840174b8a5403ad1c1255e37b8782b"\nPIPNET_REVISION = "b9eab58816437403a34aa5bc3adeafe5081fd36b"\nPYTORCH3D_REVISION = "75ebeeaea0908c5527e7b1e305fbc7681382db47"  # stable\nNVDIFFRAST_REVISION = "729261dc64c4241ea36efda84fbf532cc8b425b8"  # v0.3.3\n\n\ndef run(command, root: Path, cwd=None, env=None, step: str = "runtime", view: str = "all") -> None:\n    """Keep upstream numeric/debug output in private VM logs, never notebook outputs."""\n    effective_env = os.environ if env is None else env\n    log_root = Path(effective_env.get("DT_LOG_ROOT", str(root)))\n    directory = log_root / "logs"\n    directory.mkdir(parents=True, exist_ok=True)\n    log = directory / (step + "-" + view + ".log")\n    with log.open("a", encoding="utf-8") as stream:\n        # Keep the worker and nested stages in one group so interruption kills\n        # subprocesses before the notebook deletes their private directories.\n        new_group = os.name == "posix" and effective_env.get("DT_WORKER_ACTIVE") != "1"\n        process = subprocess.Popen([str(arg) for arg in command], cwd=cwd, env=env,\n                                   stdout=stream, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                                   start_new_session=new_group)\n        try:\n            status = process.wait()\n            if status:\n                raise subprocess.CalledProcessError(status, [str(arg) for arg in command])\n        except BaseException:\n            try:\n                if new_group:\n                    os.killpg(process.pid, signal.SIGKILL)\n                elif process.poll() is None:\n                    process.kill()\n            except ProcessLookupError:\n                pass\n            process.wait()\n            raise\n\n\ndef replace_exact(path: Path, old: str, new: str) -> None:\n    source = path.read_text(encoding="utf-8")\n    if source.count(old) != 1:\n        raise RuntimeError(f"Upstream adapter mismatch in {path.name}; refusing silent fallback")\n    path.write_text(source.replace(old, new), encoding="utf-8")\n\n\ndef replace_once_or_done(path: Path, old: str, new: str) -> None:\n    if new in path.read_text(encoding="utf-8"):\n        return\n    replace_exact(path, old, new)\n\n\ndef apply_runtime_patches(root: Path) -> None:\n    source = root / "pixel3dmm"\n    replace_once_or_done(source / "src/pixel3dmm/tracking/tracker.py", "COMPILE = True", "COMPILE = False")\n    replace_once_or_done(source / "scripts/network_inference.py", "model = model.cuda()", "model = model.eval().float().cuda()")\n\n\ndef cache_signature(architecture: str) -> dict:\n    return {"version": 1, "architecture": architecture, "pixel3dmm": PIXEL_REVISION,\n            "MICA": MICA_REVISION, "facer": FACER_REVISION, "PIPNet": PIPNET_REVISION,\n            "pytorch3d": PYTORCH3D_REVISION, "nvdiffrast": NVDIFFRAST_REVISION,\n            "torch": "2.7.1", "cuda": "11.8", "python": "3.9"}\n\n\ndef require_cache(root: Path, architecture: str, version: str = "2020") -> None:\n    marker = root / "install_complete.json"\n    if not marker.is_file():\n        raise RuntimeError("Compatible installed cache missing; run the install-and-fit cell first")\n    if json.loads(marker.read_text()) != cache_signature(architecture):\n        raise RuntimeError("Installed cache is incompatible; run final privacy cleanup with DELETE_VM_CACHE=True, then install-and-fit")\n    source = root / "pixel3dmm"\n    assets = source / "src/pixel3dmm/preprocessing/MICA/data"\n    required = [root / "env/bin/python", root / "FLAME2020.zip", assets / "FLAME2020/generic_model.pkl",\n                source / "pretrained_weights/uv.ckpt", source / "pretrained_weights/normals.ckpt",\n                assets / "pretrained/mica.tar"]\n    if version == "2023":\n        required.extend([root / "FLAME2023.zip", assets / "FLAME2023/flame2023_no_jaw.pkl"])\n    if not all(path.is_file() for path in required):\n        raise RuntimeError("Cache assets missing; run the install-and-fit cell to prepare them")\n\n\ndef runtime_environment(root: Path, architecture: str, log_root: Path) -> dict:\n    prefix = root / "env"\n    env = os.environ.copy()\n    env.update({\n        "PATH": str(prefix / "bin") + os.pathsep + env["PATH"],\n        "CONDA_PREFIX": str(prefix), "CUDA_HOME": str(prefix),\n        "LD_LIBRARY_PATH": str(prefix / "lib") + os.pathsep + env.get("LD_LIBRARY_PATH", ""),\n        "CC": str(prefix / "bin/x86_64-conda-linux-gnu-gcc"),\n        "CXX": str(prefix / "bin/x86_64-conda-linux-gnu-g++"),\n        "CUDAHOSTCXX": str(prefix / "bin/x86_64-conda-linux-gnu-g++"),\n        "TORCH_CUDA_ARCH_LIST": architecture, "MAX_JOBS": "2",\n        "TORCH_HOME": str(root / "cache/torch"), "HF_HOME": str(root / "cache/hf"),\n        "XDG_CACHE_HOME": str(root / "cache"), "MPLCONFIGDIR": str(root / "cache/matplotlib"),\n        "TORCH_EXTENSIONS_DIR": str(root / ("cache/extensions-sm" + architecture.replace(".", ""))),\n        "PIP_CACHE_DIR": str(root / "cache/pip"), "DT_INSIGHTFACE_ROOT": str(root / "insightface"),\n        "WANDB_MODE": "disabled", "HF_HUB_DISABLE_TELEMETRY": "1", "DO_NOT_TRACK": "1",\n        "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1", "TORCHDYNAMO_DISABLE": "1", "XFORMERS_DISABLED": "1",\n        "MPLBACKEND": "Agg", "DT_PYTHON": str(prefix / "bin/python"), "DT_SESSION": str(root),\n        "DT_CACHE_ROOT": str(root), "DT_LOG_ROOT": str(log_root), "PYTHONUNBUFFERED": "1",\n    })\n    constraints = root / "constraints.txt"\n    if constraints.is_file():\n        env["PIP_CONSTRAINT"] = str(constraints)\n    return env\n\n\ndef mica_assets(root: Path) -> None:\n    """Called instead of upstream\'s credential-based MICA/FLAME install script."""\n    os.environ["DT_WORKER_ACTIVE"] = "1"\n    source = root / "pixel3dmm"\n    mica = source / "src/pixel3dmm/preprocessing/MICA"\n    models = root / "insightface/models"\n    destinations = [\n        ("1bYsI_spptzyuFmfLYqYkcJA6GZWZViNt", mica / "data/pretrained/mica.tar"),\n        ("16PWKI_RjjbE4_kqpElG-YFqe8FpXjads", models / "antelopev2.zip"),\n        ("1navJMy0DTr1_DHjLWu1i48owCPvXWfYc", models / "buffalo_l.zip"),\n    ]\n    for identifier, path in destinations:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        run([sys.executable, "-m", "gdown", identifier, "-O", path], root)\n        if path.suffix == ".zip":\n            target = path.with_suffix("")\n            safe_extract(path, target)\n            # Accommodate either flat or named-folder archives expected by insightface.\n            nested = target / target.name\n            if nested.is_dir():\n                for child in nested.iterdir():\n                    shutil.move(str(child), target / child.name)\n                nested.rmdir()\n    detector = mica / "utils/landmark_detector.py"\n    replace_once_or_done(detector, "FaceAnalysis(name=\'antelopev2\', providers=[\'CUDAExecutionProvider\'])",\n                  "FaceAnalysis(name=\'antelopev2\', root=os.environ[\'DT_INSIGHTFACE_ROOT\'], providers=[\'CPUExecutionProvider\'])")\n    text = detector.read_text(encoding="utf-8")\n    if not text.startswith("import os\\n"):\n        detector.write_text("import os\\n" + text, encoding="utf-8")\n\n\ndef install(root: Path, architecture: str, log_root: Path) -> tuple:\n    """Python 3.9 + CUDA 11.8 isolated from the Colab kernel; no restart needed."""\n    env = runtime_environment(root, architecture, log_root)\n    if (root / "install_complete.json").is_file():\n        require_cache(root, architecture)\n        assets = root / "pixel3dmm/src/pixel3dmm/preprocessing/MICA/data"\n        if (root / "FLAME2023.zip").is_file() and not (assets / "FLAME2023/flame2023_no_jaw.pkl").is_file():\n            stage_flame(root / "FLAME2023.zip", assets, "2023")\n        apply_runtime_patches(root)\n        print("Reusing installed VM cache; no environment install or weight downloads.")\n        return root / "env/bin/python", env\n    print("Installing isolated Python 3.9 / CUDA 11.8; native builds can take tens of minutes.")\n    archive = root / "micromamba.tar.bz2"\n    urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/2.0.5", archive)\n    with tarfile.open(archive) as bundle:\n        member = bundle.getmember("bin/micromamba")\n        if not member.isfile():\n            raise RuntimeError("Unexpected micromamba package")\n        with bundle.extractfile(member) as binary, (root / "micromamba").open("wb") as output:\n            shutil.copyfileobj(binary, output)\n    mamba = root / "micromamba"\n    mamba.chmod(0o755)\n    prefix = root / "env"\n    cuda_packages = ["cuda-nvcc", "cuda-cccl", "cuda-cudart", "cuda-cudart-dev",\n                     "libcusparse", "libcusparse-dev", "libcublas", "libcublas-dev",\n                     "libcurand", "libcurand-dev", "libcusolver", "libcusolver-dev"]\n    operation = "install" if (prefix / "bin/python").is_file() else "create"\n    run([mamba, operation, "-y", "--no-rc", "-r", root / "mamba", "-p", prefix,\n         "-c", "conda-forge", "-c", "nvidia/label/cuda-11.8.0",\n         "python=3.9", "pip", "gcc_linux-64=11", "gxx_linux-64=11",\n         *["nvidia/label/cuda-11.8.0::" + package for package in cuda_packages]], root, env=env)\n    python = prefix / "bin/python"\n\n    def pip(*args):\n        run([python, "-m", "pip", "install", *args], root, env=env)\n\n    run(["apt-get", "update", "-qq"], root, env=env)\n    run(["apt-get", "install", "-y", "git", "build-essential", "ffmpeg", "libgl1",\n         "libegl1-mesa-dev", "libgles2-mesa-dev"], root, env=env)\n    pip("pip==25.1.1", "setuptools==68.2.2", "wheel", "numpy==1.23.5", "scipy==1.11.4", "Cython<3")\n    pip("torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",\n        "--index-url", "https://download.pytorch.org/whl/cu118")\n    source = root / "pixel3dmm"\n    if not (source / ".git").is_dir():\n        run(["git", "clone", "https://github.com/SimonGiebenhain/pixel3dmm.git", source], root, env=env)\n    else:\n        run(["git", "-C", source, "restore", "--worktree", "."], root, env=env)\n    run(["git", "-C", source, "checkout", "--detach", PIXEL_REVISION], root, env=env)\n    constraints = root / "constraints.txt"\n    constraints.write_text(\n        "numpy==1.23.5\\nscipy==1.11.4\\nscikit-image==0.22.0\\nopencv-python==4.10.0.84\\n"\n        "matplotlib==3.7.5\\npytorch-lightning==2.5.2\\ntorchmetrics==1.7.2\\n"\n        "librosa==0.10.2.post1\\nnumba==0.60.0\\ntimm==1.0.15\\n"\n        "torch==2.7.1\\ntorchvision==0.22.1\\n", encoding="utf-8")\n    env["PIP_CONSTRAINT"] = str(constraints)\n    requirements = root / "requirements.txt"\n    requirements.write_text((source / "requirements.txt").read_text().replace("numpy==1.23", "numpy==1.23.5"))\n    pip("chumpy==0.70", "--no-build-isolation")\n    pip("-r", str(requirements), "gdown==5.2.0", "trimesh>=4.6,<5", "onnxruntime==1.19.2", "ninja", "iopath")\n    for repo, revision in [("facebookresearch/pytorch3d", PYTORCH3D_REVISION), ("NVlabs/nvdiffrast", NVDIFFRAST_REVISION)]:\n        pip("git+https://github.com/" + repo + ".git@" + revision, "--no-build-isolation")\n    pip("-e", str(source))\n\n    # Execute upstream\'s installer after narrowly adapting its VM copy. Do not\n    # execute the credential prompts or its FLAME downloads at any point.\n    script = source / "install_preprocessing_pipeline.sh"\n    for old, new in [\n        ("git clone git@github.com:FacePerceiver/facer.git", "if [ ! -d facer/.git ]; then git clone https://github.com/FacePerceiver/facer.git; fi\\ngit -C facer checkout --detach " + FACER_REVISION),\n        ("git clone git@github.com:Zielon/MICA.git", "if [ ! -d MICA/.git ]; then git clone https://github.com/Zielon/MICA.git; fi\\ngit -C MICA checkout --detach " + MICA_REVISION),\n        ("git clone https://github.com/jhb86253817/PIPNet.git", "if [ ! -d PIPNet/.git ]; then git clone https://github.com/jhb86253817/PIPNet.git; fi\\ngit -C PIPNet checkout --detach " + PIPNET_REVISION),\n    ]:\n        replace_exact(script, old, new)\n    script.write_text("#!/bin/bash\\nset -euo pipefail\\n" + script.read_text().replace("gdown --id ", "gdown ").replace("mkdir ", "mkdir -p "))\n    replacement = source / "src/pixel3dmm/preprocessing/replacement_code/install_mica_download_flame.sh"\n    replacement.write_text(\'#!/bin/bash\\nset -euo pipefail\\nexec "$DT_PYTHON" "$DT_SESSION/helpers/setup_runtime.py" mica-assets "$DT_SESSION"\\n\')\n    run(["bash", script], root, cwd=source, env=env)\n    assets = source / "src/pixel3dmm/preprocessing/MICA/data"\n    stage_flame(root / "FLAME2020.zip", assets, "2020")\n    if (root / "FLAME2023.zip").exists():\n        stage_flame(root / "FLAME2023.zip", assets, "2023")\n    apply_runtime_patches(root)\n    run([python, "-m", "pip", "check"], root, env=env)\n    run([python, "-c", "import torch, pytorch3d, nvdiffrast.torch, facer, insightface; assert torch.cuda.is_available()"], root, env=env)\n    (root / "install_complete.json").write_text(json.dumps(cache_signature(architecture)), encoding="utf-8")\n    return python, env\n\n\nif __name__ == "__main__":\n    if sys.argv[1] != "mica-assets":\n        raise ValueError("Unknown installation operation")\n    mica_assets(Path(sys.argv[2]))\n', 'worker.py': '"""User-operated Colab multi-photo fit. Never run on personal data during QA."""\n\nfrom pathlib import Path\nimport json\nimport os\nimport re\nimport shutil\nimport sys\nimport subprocess\n\nfrom io_utils import upload_views\nfrom setup_runtime import run\nfrom diagnostics import step_context\n\n\nclass DetectionError(RuntimeError):\n    """A view has no usable face/landmarks; optional views may be omitted."""\n\n\ndef is_detection_failure(log: Path) -> bool:\n    text = log.read_text(encoding="utf-8", errors="replace") if log.is_file() else ""\n    exceptions = [line.strip() for line in text.splitlines()\n                  if re.match(r"^[\\w.]+(?:Error|Exception):", line.strip())]\n    # A prior detection warning must never mask a later CUDA/import exception.\n    terminal = exceptions[-1] if exceptions else ""\n    return any(message in terminal for message in (\n        "IndexError: list index out of range", "ValueError: need at least one array to stack",\n        "Found face with too low detections confidence", "Face not detected",\n    ))\n\n\ndef skip_optional(root: Path, view: str, reason: str) -> dict:\n    context = json.loads((root / "current_step.json").read_text())\n    if view == "front":\n        raise DetectionError("Front face required; retry with a clear frontal photo")\n    item = {"view": view, "reason": reason, "step": context["step"]}\n    path = root / "warnings.json"\n    warnings = json.loads(path.read_text()) if path.is_file() else []\n    warnings.append(item)\n    path.write_text(json.dumps(warnings), encoding="utf-8")\n    return item\n\n\ndef run_step(root: Path, script: Path, args: list, step: str, view: str = "all", cwd=None) -> None:\n    step_context(root, step, view)\n    wrapper = Path(__file__).parent / "runtime_compat.py"\n    try:\n        run([sys.executable, wrapper, script, *args], root, cwd=cwd, step=step, view=view)\n    except subprocess.CalledProcessError as error:\n        if step == "cropping_landmarks" and is_detection_failure(root / f"logs/{step}-{view}.log"):\n            raise DetectionError("No usable crop or facial landmarks") from error\n        raise\n\n\ndef preprocess(root: Path) -> list:\n    step_context(root, "detector_initialization")\n    import numpy as np\n    from PIL import Image, ImageOps\n    from insightface.app import FaceAnalysis\n\n    source = Path(os.environ["DT_CACHE_ROOT"]) / "pixel3dmm"\n    data = root / "preprocessed"\n    merged = data / "head"\n    views = upload_views(path.name for path in (root / "uploads").iterdir())\n    included, skipped = [], []\n    detector = FaceAnalysis(name="antelopev2", root=os.environ["DT_INSIGHTFACE_ROOT"],\n                            allowed_modules=["detection"], providers=["CPUExecutionProvider"])\n    detector.prepare(ctx_id=-1, det_size=(512, 512))\n    for view in ("front", "left", "right", "back"):\n        if view not in views:\n            skipped.append({"view": view, "reason": "not_uploaded"})\n            continue\n        step_context(root, "face_detection", view)\n        input_dir = root / "inputs" / view\n        input_dir.mkdir(parents=True)\n        with Image.open(root / "uploads" / views[view]) as image:\n            normalized = ImageOps.exif_transpose(image).convert("RGB")\n            original_size = list(normalized.size)\n            normalized.save(input_dir / "input.png")\n            # Insightface receives BGR, matching its documented API.\n            if len(detector.get(np.asarray(normalized)[:, :, ::-1].copy())) == 0:\n                skipped.append(skip_optional(root, view, "no_face_detected"))\n                continue\n        # run_preprocessing.py uses unchecked os.system. Execute its three\n        # component scripts with checked subprocesses, independently per view.\n        try:\n            run_step(root, source / "scripts/run_cropping.py", ["--video_or_images_path", input_dir],\n                     "cropping_landmarks", view, cwd=source)\n        except DetectionError:\n            skipped.append(skip_optional(root, view, "no_usable_crop_or_landmarks"))\n            continue\n        folder = data / view\n        crop = folder / "cropped/00000.jpg"\n        landmark = folder / "PIPnet_landmarks/00000.npy"\n        bounds = folder / "crop_ymin_ymax_xmin_xmax.npy"\n        if not all(path.is_file() for path in (crop, landmark, bounds)):\n            skipped.append(skip_optional(root, view, "crop_or_landmarks_missing"))\n            continue\n        lm = np.load(landmark, allow_pickle=False)\n        if lm.shape != (98, 2) or not np.isfinite(lm).all() or not lm.any():\n            skipped.append(skip_optional(root, view, "invalid_facial_landmarks"))\n            continue\n        run_step(root, source / "src/pixel3dmm/preprocessing/MICA/demo.py",\n                 ["-video_name", view, "-a", folder / "arcface"], "mica", view,\n                 cwd=source / "src/pixel3dmm/preprocessing/MICA")\n        identity = folder / "mica/00000/identity.npy"\n        if not identity.is_file():\n            skipped.append(skip_optional(root, view, "mica_face_output_missing"))\n            continue\n        run_step(root, source / "scripts/run_facer_segmentation.py", ["--video_name", view],\n                 "segmentation", view, cwd=source)\n        segment = folder / "seg_og/00000.png"\n        if not segment.is_file():\n            skipped.append(skip_optional(root, view, "face_segmentation_output_missing"))\n            continue\n        index = len(included)\n        for path, destination in [\n            (crop, merged / f"cropped/{index:05d}.jpg"),\n            (landmark, merged / f"PIPnet_landmarks/{index:05d}.npy"),\n            (segment, merged / f"seg_og/{index:05d}.png"),\n        ]:\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(path, destination)\n        shutil.copytree(identity.parent, merged / f"mica/{index:05d}")\n        included.append({"view": view, "frame": index, "originalSizeWH": original_size,\n                         "cropBoundsYminYmaxXminXmax": np.load(bounds, allow_pickle=False).tolist()})\n    if not any(item["view"] == "front" for item in included):\n        raise DetectionError("Front must preprocess successfully")\n    (root / "view_map.json").write_text(json.dumps({"included": included, "skipped": skipped}, indent=2))\n    for prediction in ("normals", "uv_map"):\n        run_step(root, source / "scripts/network_inference.py",\n                 ["model.prediction_type=" + prediction, "video_name=head", "viz_uv_mesh=False"],\n                 prediction + "_prediction", cwd=source)\n        for item in included:\n            path = merged / f"p3dmm/{prediction}/{item[\'frame\']:05d}.png"\n            if not path.is_file():\n                step_context(root, prediction + "_prediction", item["view"])\n                raise RuntimeError("Pixel3DMM prediction missing; upstream may have caught an inference error")\n    return included\n\n\ndef fit(root: Path, included: list, config: dict) -> None:\n    step_context(root, "tracking")\n    import torch\n    from omegaconf import OmegaConf\n    from pixel3dmm.tracking.tracker import Tracker\n\n    source = Path(os.environ["DT_CACHE_ROOT"]) / "pixel3dmm"\n    overrides = {\n        "video_name": "head", "num_views": 1, "batch_size": min(len(included), config["max_fit_batch_size"]),\n        "iters": config["iters"], "global_iters": config["global_iters"],\n        "is_discontinuous": True, "global_camera": False, "include_neck": False,\n        "use_flame2023": config["flame_version"] == "2023",\n        "ignore_mica": config["flame_version"] == "2023",\n        "w_exp": 0.1, "use_mouth_lmk": False, "w_shape": 0.01, "w_shape_general": 0.001,\n        "normal_super": 2000.0, "sil_super": 1000.0,\n        "save_meshes": True, "save_landmarks": False, "delete_preprocessing": False,\n        "size": 256, "image_size": [256, 256],\n    }\n    cfg = OmegaConf.merge(OmegaConf.load(source / "configs/tracking.yaml"), overrides)\n    tracker = Tracker(cfg)\n    tracker.run()\n    # Explicitly save every view AFTER joint optimization. canonical.ply saved\n    # during online initialization is stale and must not be used as neutral export.\n    for item in included:\n        index = item["frame"]\n        tracker.save_checkpoint(index, selected_frames=torch.tensor([index], device="cuda"))\n    (root / "fit_path.txt").write_text(str(Path(tracker.output_folder)))\n    (root / "fit_config.json").write_text(json.dumps(OmegaConf.to_container(cfg, resolve=True)))\n    tracker.writer.close()\n\n\ndef main(root: Path) -> None:\n    os.environ["DT_WORKER_ACTIVE"] = "1"\n    step_context(root, "runtime_configuration")\n    from runtime_compat import configure\n    configure()\n    import torch\n    with torch.autocast(device_type="cuda", enabled=False):\n        config = json.loads((root / "config.json").read_text())\n        included = preprocess(root)\n        fit(root, included, config)\n        step_context(root, "export")\n        from export_fit import export_fit\n        export_fit(root, config)\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n', 'export_fit.py': '"""Export fitted identity, neutral geometry, native cameras and alignment overlays."""\n\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport json\nimport shutil\nimport zipfile\n\nimport numpy as np\n\nfrom camera import camera_matrices, projection_matrix\nfrom setup_runtime import PIXEL_REVISION, MICA_REVISION, FACER_REVISION, PIPNET_REVISION\n\n\ndef render_overlay(vertices, faces, camera, image, destination: Path) -> None:\n    import torch\n    import nvdiffrast.torch as dr\n    from PIL import Image\n    import trimesh\n\n    intrinsics, extrinsics, side = camera\n    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)\n    points = torch.tensor(np.column_stack([vertices, np.ones(len(vertices))]), dtype=torch.float32, device="cuda")\n    matrix = torch.tensor(projection_matrix(intrinsics, side) @ extrinsics, dtype=torch.float32, device="cuda")\n    clips = (points @ matrix.T)[None].contiguous()\n    triangles = torch.tensor(faces.astype(np.int32), device="cuda").contiguous()\n    context = dr.RasterizeCudaContext()\n    raster, _ = dr.rasterize(context, clips, triangles, resolution=[side, side])\n    normals = torch.tensor(np.asarray(mesh.vertex_normals).copy(), dtype=torch.float32, device="cuda")[None]\n    normal_image, _ = dr.interpolate(normals.contiguous(), raster, triangles)\n    shading = (0.35 + 0.65 * normal_image[..., 2:3].abs()).clamp(0, 1)\n    color = shading * torch.tensor([0.2, 0.7, 1.0], device="cuda")\n    mask = (raster[..., 3:4] > 0).float()\n    # The upstream projection matrix has negative fy; array row 0 is already top.\n    rendered = (color * mask)[0].cpu().numpy()\n    alpha = mask[0].cpu().numpy() * 0.5\n    background = np.asarray(Image.open(image).convert("RGB").resize((side, side)), dtype=np.float32) / 255\n    Image.fromarray(np.clip((background * (1 - alpha) + rendered * 0.5) * 255, 0, 255).astype(np.uint8)).save(destination)\n\n\ndef export_fit(root: Path, config: dict) -> None:\n    import torch\n    import trimesh\n    from omegaconf import OmegaConf\n    from pixel3dmm.tracking.flame.FLAME import FLAME\n\n    fit_path = Path((root / "fit_path.txt").read_text())\n    view_map = json.loads((root / "view_map.json").read_text())\n    out = root / "export"\n    (out / "overlays").mkdir(parents=True)\n    (out / "fitted_views").mkdir()\n    checkpoints = [torch.load(fit_path / f"checkpoint/{item[\'frame\']:05d}.frame", map_location="cpu", weights_only=False)\n                   for item in view_map["included"]]\n    shape = np.asarray(checkpoints[0]["flame"]["shape"], dtype=np.float32).reshape(1, -1)\n    if shape.shape != (1, 300) or not np.isfinite(shape).all():\n        raise ValueError("Invalid shared shape coefficients")\n    for checkpoint in checkpoints:\n        if not np.allclose(shape, checkpoint["flame"]["shape"]):\n            raise ValueError("Final per-view checkpoints do not share identity")\n    flame = FLAME(OmegaConf.create({"num_shape_params": 300, "num_exp_params": 100,\n                                   "use_flame2023": config["flame_version"] == "2023"})).cuda()\n    with torch.no_grad():\n        vertices = flame(shape_params=torch.tensor(shape, device="cuda"), cameras=None)[0][0].cpu().numpy()\n    faces = flame.faces.cpu().numpy()\n    neutral = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)\n    neutral.export(out / "head_neutral.obj")\n    neutral.export(out / "head_neutral.ply")\n    identity6d = [1, 0, 0, 0, 1, 0]\n    neutral_parameters = {"shape": shape[0].tolist(), "exp": [0.0] * 100,\n                          "R": identity6d, "t": [0.0] * 3, "neck": identity6d,\n                          "jaw": identity6d, "eyes": identity6d * 2, "eyelids": [0.0, 0.0]}\n    parameters = {"schema": "dt-flame-head-parameters/1", "flameVersion": config["flame_version"],\n                  "rotationEncoding": "6D: first two ROWS of a rotation matrix (Pixel3DMM/PyTorch3D)",\n                  "neutral": neutral_parameters, "views": {}}\n    arrays = {"neutral_" + key: np.asarray(value, dtype=np.float32) for key, value in neutral_parameters.items()}\n    cameras = {"schema": "dt-flame-head-cameras/1", "model": "OpenGL perspective pinhole",\n               "grid": "256x256 independent face crop; pixel centers at (column+0.5,row+0.5)",\n               "projection": "q = worldToCamera @ [x,y,z,1]; depth=-q.z; u=fx*q.x/depth+cx; v=cy-fy*q.y/depth",\n               "meshSpace": "native FLAME metres, unnormalized; extrinsics apply to fitted_views meshes with local neck/jaw/expression already applied",\n               "views": {}, "skipped": view_map["skipped"]}\n    for item, checkpoint in zip(view_map["included"], checkpoints):\n        view, index = item["view"], item["frame"]\n        fitted = {key: np.asarray(value) for key, value in checkpoint["flame"].items()}\n        if not all(np.isfinite(value).all() for value in fitted.values()):\n            raise ValueError("Nonfinite fitted parameters")\n        parameters["views"][view] = {key: value.tolist() for key, value in fitted.items()}\n        arrays.update({view + "_" + key: value for key, value in fitted.items()})\n        joint_transforms = np.asarray(checkpoint["joint_transforms"])\n        arrays[view + "_joint_transforms"] = joint_transforms\n        intrinsics, extrinsics, side = camera_matrices(checkpoint)\n        # A rigid head-centric camera cannot undo nonrigid expression/jaw changes.\n        # Preserve the local neck transform separately, as upstream visualization does.\n        head_centric = extrinsics @ joint_transforms[0, 1]\n        cameras["views"][view] = {\n            **item, "imageSizeWH": [side, side], "intrinsics": intrinsics.tolist(),\n            "worldToCamera": extrinsics.tolist(), "headCentricWorldToCamera": head_centric.tolist(),\n            "headCentricNote": "Use only with the matching fitted mesh after undoing joint_transforms[0,1]; neutral expression changes geometry",\n            "native": {key: np.asarray(value).tolist() for key, value in checkpoint["camera"].items()},\n            "projectionMatrix": projection_matrix(intrinsics, side).tolist(),\n            "overlay": f"overlays/{view}.png", "fittedMesh": f"fitted_views/{view}.ply",\n        }\n        arrays[view + "_intrinsics"] = intrinsics\n        arrays[view + "_world_to_camera"] = extrinsics\n        source_mesh = fit_path / f"mesh/{index:05d}.ply"\n        mesh = trimesh.load(source_mesh, process=False, force="mesh")\n        shutil.copyfile(source_mesh, out / f"fitted_views/{view}.ply")\n        render_overlay(np.asarray(mesh.vertices), np.asarray(mesh.faces), (intrinsics, extrinsics, side),\n                       root / f"preprocessed/head/cropped/{index:05d}.jpg", out / f"overlays/{view}.png")\n    (out / "parameters.json").write_text(json.dumps(parameters, indent=2, allow_nan=False))\n    np.savez_compressed(out / "parameters.npz", **arrays)\n    (out / "cameras.json").write_text(json.dumps(cameras, indent=2, allow_nan=False))\n    provenance = {"createdAt": datetime.now(timezone.utc).isoformat(), "flameVersion": config["flame_version"],\n                  "sources": {"pixel3dmm": PIXEL_REVISION, "MICA": MICA_REVISION, "facer": FACER_REVISION, "PIPNet": PIPNET_REVISION},\n                  "license": "Personal non-commercial research only; Pixel3DMM CC BY-NC 4.0 plus separately accepted FLAME/MICA/insightface/nvdiffrast terms. Not cleared for app distribution.",\n                  "isDiscontinuous": True, "globalCamera": False, "neutralExpression": True, "neutralJaw": True,\n                  "includedViews": [item["view"] for item in view_map["included"]], "skippedViews": view_map["skipped"],\n                  "fitSettings": json.loads((root / "fit_config.json").read_text()),\n                  "limitations": ["No hair reconstruction", "Unseen back anatomy follows FLAME prior", "Scale from model prior; no metric calibration", "Not the rigged twin.glb bundle"]}\n    (out / "provenance.json").write_text(json.dumps(provenance, indent=2, allow_nan=False))\n    with zipfile.ZipFile(root / "head_fit.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(out.rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(out).as_posix())\n', 'diagnostics.py': '"""Show bounded stack traces, never raw upstream image/array/debug output."""\n\nfrom collections import deque\nfrom pathlib import Path\nimport json\nimport re\n\n\ndef traceback_tail(path: Path, limit: int = 120) -> list:\n    if not path.is_file():\n        return []\n    # Upstream prints detections, arrays and progress bars. Never dump raw logs.\n    with path.open(encoding="utf-8", errors="replace") as stream:\n        lines = deque(stream, maxlen=8000)\n    result = []\n    for raw in lines:\n        line = re.sub(r"\\x1b\\[[0-9;]*[A-Za-z]", "", raw).strip()\n        if line.startswith(("Traceback (", "During handling of the above exception", "The above exception")):\n            result.append(line)\n        elif re.fullmatch(r\'File "[^"\\n]+", line \\d+(?:, in [\\w<>.]+)?\', line):\n            result.append("  " + line)\n        elif re.match(r"^(?:[\\w.]+(?:Error|Exception)|KeyboardInterrupt|SystemExit|AssertionError):", line):\n            kind, _, message = line.partition(":")\n            # Do not print array/tensor/bytes reprs, long payloads, numeric dumps,\n            # credentials, or arbitrary strings embedded in an exception message.\n            unsafe = (len(message) > 600 or bool(re.search(\n                r"array\\(|tensor\\(|\\[|\\{|data:image|base64|\\\\x[0-9a-fA-F]{2}|"\n                r"(?i:password|credential|token\\s*[=:])|(?:\\d[., ]+){8}", message)))\n            if unsafe:\n                message = " [message payload omitted]"\n            else:\n                message = re.sub(r"([\'\\"]).*?\\1", "<quoted value>", message)\n            result.append(kind + ":" + message)\n    return result[-limit:]\n\n\ndef step_context(root: Path, step: str, view: str = "all") -> None:\n    (root / "current_step.json").write_text(json.dumps({"step": step, "view": view}), encoding="utf-8")\n\n\ndef print_diagnostics(root: Path, limit: int = 120) -> None:\n    context = root / "current_step.json"\n    if context.is_file():\n        info = json.loads(context.read_text(encoding="utf-8"))\n        # Values come from our labels, but still exclude arbitrary user input.\n        step = re.sub(r"[^a-zA-Z0-9_ -]", "?", str(info.get("step", "unknown")))[:80]\n        view = info.get("view", "all")\n        if view not in {"all", "front", "left", "right", "back"}:\n            view = "all"\n        print(f"Failed step: {step}; view: {view}")\n    traces = []\n    logs = sorted((root / "logs").glob("*.log"), key=lambda path: path.stat().st_mtime)\n    # Per-step logs contain the underlying exception, worker.log its calling stack.\n    for path in logs:\n        tail = traceback_tail(path, limit)\n        if tail:\n            traces.extend(["Traceback log: " + path.name, *tail])\n    print(f"Last {limit} filtered traceback lines (raw image/debug output omitted):")\n    print("\\n".join(traces[-limit:]) if traces else "No Python stack trace captured; the process may have been killed or failed in native code.")\n\n\ndef print_warnings(root: Path) -> None:\n    path = root / "warnings.json"\n    if not path.is_file():\n        return\n    for item in json.loads(path.read_text(encoding="utf-8")):\n        if item.get("view") in {"left", "right", "back"}:\n            reason = re.sub(r"[^a-zA-Z0-9_ -]", "?", str(item.get("reason", "preprocessing_failed")))[:80]\n            step = re.sub(r"[^a-zA-Z0-9_ -]", "?", str(item.get("step", "detection")))[:80]\n            print(f"Warning: skipped {item[\'view\']} at {step}: {reason}.")\n', 'runtime_compat.py': '"""Original T4-safe execution policy, shared by the worker and preprocessing."""\n\nimport os\n\n\ndef configure() -> None:\n    os.environ["XFORMERS_DISABLED"] = "1"\n    os.environ["TORCHDYNAMO_DISABLE"] = "1"\n    import torch\n    from timm.layers import set_fused_attn\n\n    torch.set_default_dtype(torch.float32)\n    torch.set_autocast_dtype("cuda", torch.float16)  # No T4-incompatible bf16 default.\n    torch.backends.cuda.enable_flash_sdp(False)\n    torch.backends.cuda.enable_mem_efficient_sdp(False)\n    torch.backends.cuda.enable_math_sdp(True)\n    torch.backends.cuda.enable_cudnn_sdp(False)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    set_fused_attn(False)  # timm DINO uses its ordinary matmul/softmax attention.\n\n\nif __name__ == "__main__":\n    import runpy\n    import sys\n    from pathlib import Path\n\n    configure()\n    script = Path(sys.argv[1]).resolve()\n    sys.argv = [str(script), *sys.argv[2:]]\n    sys.path.insert(0, str(script.parent))\n    import torch\n    with torch.autocast(device_type="cuda", enabled=False):\n        runpy.run_path(str(script), run_name="__main__")\n', 'notebook_session.py': '"""Separate persistent software/model cache from every private photo session."""\n\nfrom pathlib import Path\nimport gc\nimport json\nimport shutil\nimport tempfile\n\nfrom diagnostics import print_diagnostics, print_warnings, step_context\nfrom io_utils import cleanup_session, download_and_wait, upload_views\nfrom setup_runtime import apply_runtime_patches, install, require_cache, run, runtime_environment\n\n\ndef private_config(path: Path, cache: Path, parent: Path = Path("/content")) -> bool:\n    if not path.is_file():\n        return False\n    values = {}\n    for line in path.read_text(encoding="utf-8").splitlines():\n        key, separator, value = line.partition("=")\n        if separator:\n            try:\n                values[key] = json.loads(value)\n            except json.JSONDecodeError:\n                return False\n    if values.get("PIXEL3DMM_CODE_BASE") != str(cache / "pixel3dmm"):\n        return False\n    for key in ("PIXEL3DMM_PREPROCESSED_DATA", "PIXEL3DMM_TRACKING_OUTPUT"):\n        value = values.get(key)\n        if not isinstance(value, str):\n            return False\n        directory = Path(value)\n        session = directory.parent\n        if session.parent != parent or not session.name.startswith("dt-pixel3dmm-session-"):\n            return False\n    return True\n\n\ndef copy_flame_archives(cache: Path, selected: dict) -> None:\n    from google.colab import drive\n\n    record = cache / "selected_archives.json"\n    previous = json.loads(record.read_text()) if record.is_file() else {}\n    missing = {version: filename for version, filename in selected.items()\n               if not (cache / ("FLAME" + version + ".zip")).is_file() or previous.get(version) != filename}\n    if not missing:\n        print("Reusing copied FLAME archives; no Drive mount needed.")\n        return\n    drive.mount("/content/drive")\n    try:\n        for version, filename in missing.items():\n            # Preserve the lead\'s FUSE-safe check, relative paths and default flame/ folder.\n            raw = str(filename).strip().strip(\'"\').strip("\'")\n            model_zip = Path(raw) if raw.startswith("/") else Path("/content") / raw\n            if not str(model_zip).startswith("/content/drive/") or ".." in model_zip.parts or model_zip.suffix.lower() != ".zip":\n                raise ValueError(f"Select a FLAME .zip inside /content/drive (got {raw!r})")\n            if not model_zip.is_file():\n                raise FileNotFoundError("FLAME zip not found; check the parameter cell")\n            # Read-only Drive usage: the only Drive file operation is this copy.\n            destination = cache / ("FLAME" + version + ".zip")\n            temporary = destination.with_suffix(".zip.partial")\n            try:\n                shutil.copyfile(model_zip, temporary)\n                temporary.replace(destination)\n            finally:\n                temporary.unlink(missing_ok=True)\n            previous[version] = filename\n            # A changed user model must be restaged even with a ready environment.\n            stage_model(cache, version)\n        record.write_text(json.dumps(previous), encoding="utf-8")\n    finally:\n        drive.flush_and_unmount()\n\n\ndef stage_model(cache: Path, version: str) -> None:\n    from io_utils import stage_flame\n    assets = cache / "pixel3dmm/src/pixel3dmm/preprocessing/MICA/data"\n    if assets.is_dir():\n        stage_flame(cache / ("FLAME" + version + ".zip"), assets, version)\n\n\ndef run_session(cache: Path, selected: dict, version: str, architecture: str, iters: int,\n                global_iters: int, max_fit_batch_size: int, prepare: bool = False) -> None:\n    from google.colab import files\n\n    if version not in {"2020", "2023"} or any(not isinstance(value, int) or value < 1\n                                              for value in (iters, global_iters, max_fit_batch_size)):\n        raise ValueError("Invalid model variant or iteration/batch parameters")\n\n    session = Path(tempfile.mkdtemp(prefix="dt-pixel3dmm-session-", dir="/content"))\n    uploaded = {}\n    config_created = False\n    env_file = Path.home() / ".config/pixel3dmm/.env"\n    try:\n        step_context(session, "installation" if prepare else "cache_validation")\n        if prepare:\n            copy_flame_archives(cache, selected)\n            python, worker_env = install(cache, architecture, session)\n        else:\n            require_cache(cache, architecture, version)\n            apply_runtime_patches(cache)\n            python = cache / "env/bin/python"\n            worker_env = runtime_environment(cache, architecture, session)\n            print("Fit-only retry: reusing the installed VM cache and copied FLAME files.")\n        require_cache(cache, architecture, version)\n        if env_file.exists():\n            if private_config(env_file, cache):\n                env_file.unlink()\n            else:\n                raise RuntimeError("Unrelated Pixel3DMM config found; use a fresh runtime")\n        env_file.parent.mkdir(parents=True, exist_ok=True)\n        paths = {"PIXEL3DMM_CODE_BASE": str(cache / "pixel3dmm"),\n                 "PIXEL3DMM_PREPROCESSED_DATA": str(session / "preprocessed"),\n                 "PIXEL3DMM_TRACKING_OUTPUT": str(session / "tracking")}\n        env_file.write_text("".join(key + "=" + json.dumps(value) + "\\n" for key, value in paths.items()), encoding="utf-8")\n        config_created = True\n        worker_env.update(paths)\n        (session / "config.json").write_text(json.dumps({\n            "flame_version": version, "iters": iters, "global_iters": global_iters,\n            "max_fit_batch_size": max_fit_batch_size,\n        }), encoding="utf-8")\n        uploads = session / "uploads"\n        uploads.mkdir()\n        step_context(session, "photo_upload")\n        print("Upload front and optional left/right/back head photos together.")\n        uploaded = files.upload(target_dir=str(uploads))\n        upload_views(uploaded.keys())\n        uploaded.clear()\n        print("Preprocessing each view, then fitting shared identity with independent cameras.")\n        run([python, cache / "helpers/worker.py", session], session,\n            cwd=cache / "pixel3dmm", env=worker_env, step="worker")\n        print_warnings(session)\n        step_context(session, "archive_download")\n        download_and_wait(session / "head_fit.zip")\n        print("head_fit.zip transferred to your browser. Save it to user-data/twin/head/flame/.")\n    except BaseException:\n        # These diagnostics must run while per-step logs still exist.\n        print_warnings(session)\n        print_diagnostics(session)\n        raise\n    finally:\n        uploaded.clear()\n        try:\n            if config_created:\n                env_file.unlink(missing_ok=True)\n        finally:\n            cleanup_session(session)\n            print("Privacy cleanup: uploads, derived outputs, logs and archive deleted; VM model cache retained for retry.")\n\n\ndef cleanup_everything(cache: Path, parent: Path = Path("/content"), config_file=None) -> None:\n    if cache.resolve().parent != parent.resolve() or cache.name != "dt-pixel3dmm-cache" or cache.is_symlink():\n        raise ValueError("Refusing unsafe cache cleanup")\n    for session in parent.glob("dt-pixel3dmm-session-*"):\n        cleanup_session(session, parent)\n    config = Path.home() / ".config/pixel3dmm/.env" if config_file is None else config_file\n    if private_config(config, cache, parent):\n        config.unlink()\n    if cache.exists():\n        shutil.rmtree(cache)\n    gc.collect()\n    print("Final privacy cleanup: all session files, installed environment, sources, weights and FLAME removed.")\n'}


## Copy FLAME, install, upload, fit and download

Drive authorization is only needed to read the model zip(s). No FLAME
username/password is requested. The credential sections of upstream's
preprocessing installer are replaced in the VM with model-weight downloads.
Worker stdout/stderr and each preprocessing step go to private session logs.
Failures print the failed step/view and the last 120 filtered traceback lines
BEFORE cleanup. Numeric arrays, image bytes and arbitrary debug output are excluded.
Clear saved outputs before sharing. The installed cache survives success/failure;
only photos, derived outputs and logs are automatically removed.

Each view is cropped separately before the joint fit. `is_discontinuous=True`
disables temporal smoothness; `global_camera=False` fits intrinsics per photo.
Overlays are at **256x256 on each face crop**, not original-photo coordinates.
Final shared shape coefficients generate the neutral mesh anew after joint
fitting: expression/eyelids zero, jaw/neck/eyes/global rotation identity,
global translation zero. Do not use upstream's earlier canonical mesh.

Enable browser downloads. The cell waits for `files.download()` to transfer
the archive into browser memory before deleting the VM source. It cannot
verify that you saved the browser download to disk.


In [ ]:
from pathlib import Path
import importlib
import sys

CACHE_ROOT = Path("/content/dt-pixel3dmm-cache")
if CACHE_ROOT.is_symlink() or CACHE_ROOT.resolve().parent != Path("/content"):
    raise ValueError("Refusing unsafe VM cache path")
(CACHE_ROOT / "helpers").mkdir(parents=True, exist_ok=True)
for name, source in EMBEDDED_FILES.items():
    (CACHE_ROOT / "helpers" / name).write_text(source, encoding="utf-8")
    sys.modules.pop(Path(name).stem, None)
helper_path = str(CACHE_ROOT / "helpers")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)
importlib.invalidate_caches()
from notebook_session import run_session as run_head_session, cleanup_everything

def run_session(prepare=False):
    selected = {FLAME_VERSION: FLAME_ZIP_DRIVE_PATH}
    if FLAME_VERSION == "2023":
        selected["2020"] = FLAME2020_ZIP_DRIVE_PATH
    run_head_session(CACHE_ROOT, selected, FLAME_VERSION, GPU_ARCH,
                     ITERS, GLOBAL_ITERS, MAX_FIT_BATCH_SIZE, prepare=prepare)


In [ ]:
# Install once (or reuse a ready cache), then ask for photos and fit.
_SKIP_AUTOMATIC_RETRY = False
run_session(prepare=True)
_SKIP_AUTOMATIC_RETRY = True


## Fit only: retry without installation or Drive access

After a failed fit, run the next cell to upload the photos again and retry
with the same installed environment, weights and FLAME files. You may edit
iteration/batch parameters first. A missing cache requires install-and-fit;
an incompatible cache requires final privacy cleanup before reinstalling.
During a successful Run all, this cell skips a
duplicate upload; run it again manually for another fit. Even a front-only
fit is accepted when optional profiles cannot be detected; metadata and
warnings record every omission.


In [ ]:
if globals().pop("_SKIP_AUTOMATIC_RETRY", False):
    print("Automatic duplicate fit skipped. Run this cell again for a fit-only retry.")
else:
    run_session(prepare=False)


## Final privacy cleanup

Every fit already deletes uploads, outputs, logs and its archive. Keep the
model cache while retrying. **When finished, set DELETE_VM_CACHE=True below
and run this cell** to delete everything, including installed software,
copied/extracted FLAME and downloaded weights. Default False lets Run all
retain the cache for retries. Then disconnect and delete the runtime.
No Drive files are deleted. If an earlier cell stops, run this cell manually
with True when you are done. Clear saved diagnostic outputs before sharing.


In [ ]:
DELETE_VM_CACHE = False  # Set True and run this cell once finished retrying.
if DELETE_VM_CACHE:
    cleanup_everything(CACHE_ROOT)
else:
    print("VM model cache retained for fit-only retries. Set DELETE_VM_CACHE=True here when finished.")
